In [1]:
import os

In [2]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject'

In [5]:
# Step-1: Update config.yaml
# step-2: Update params.yaml (if any)
# step-3: Update schema.yaml (if any)

# step-4: Update entity --> data class creation
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

# step-5: Update the configuration manager --> read all yaml files
from src.my_first_end_to_end_project.constants import *
from src.my_first_end_to_end_project.utils.common_utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
            self,
            config_file_path = CONFIG_FILE_PATH, 
            params_fie_path = PARAMS_FILE_PATH,
            schema_file_path = SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_fie_path)
        self.schema = read_yaml(schema_file_path)
        create_directories([self.config.artifacts_root])

    
    def get_model_trainer_config(self)-> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COUMN
        # creating the directory for models store
        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir= config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            alpha = params.alpha,
            l1_ratio= params.l1_ratio,
            target_column= schema.name
        )

        return model_trainer_config
        


# Step-6: Update components --> load split data and run model

import pandas as pd
import os
from src.my_first_end_to_end_project.logger import logger
from sklearn.linear_model import ElasticNet
import joblib

class ModelTrainer:
    def __init__(self, config:ModelTrainerConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop([self.config.target_column],axis =1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]

        lr = ElasticNet(alpha = self.config.alpha, l1_ratio = self.config.l1_ratio, random_state= 42)
        lr.fit(train_x,train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

In [6]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    logger.exception(e)
    raise e

2026-01-07 05:19:30 - INFO - common_utils - yaml file: config/config.yaml is loaded successfully 🥳
2026-01-07 05:19:30 - INFO - common_utils - yaml file: params.yaml is loaded successfully 🥳
2026-01-07 05:19:30 - INFO - common_utils - yaml file: schema.yaml is loaded successfully 🥳
2026-01-07 05:19:30 - INFO - common_utils - Created directory Successfuly at: artifacts 🥳
2026-01-07 05:19:30 - INFO - common_utils - Created directory Successfuly at: artifacts/model_trainer 🥳
